# Build the Anglican library index on HPE Private Cloud AI

Runs the resumable embedding pipeline (Qwen3-Embedding-0.6B @ 512-d -> FAISS) on a GPU
notebook workspace. **Before running:** clone this repo into the workspace, open this
notebook from the repo root, and upload `library.db` next to it.


## 1. Confirm the GPU


In [ ]:
!nvidia-smi

## 2. Install dependencies

`uv sync` builds an isolated env with the correct CUDA 12.8 PyTorch build (per `pyproject.toml`).


In [ ]:
!pip install -q uv
!uv sync

## 3. Verify PyTorch sees the GPU


In [ ]:
!uv run python scripts/check_env.py

## 4. Check the data is present

`library.db` must be uploaded into this folder (it holds the cleaned chunks + metadata).


In [ ]:
import os
assert os.path.exists('library.db'), 'Upload library.db into this folder first.'
print('library.db:', round(os.path.getsize('library.db')/1e9, 2), 'GB')

## 5. Build the index (resumable)

Embeds ~1.23M chunks. Re-running this cell resumes from where it left off if interrupted.
Raise `--encode-batch` if the GPU is underused (an H100 NVL has plenty of memory).


In [ ]:
import os
os.environ['ANGLICAN_DB'] = 'library.db'
os.environ['ANGLICAN_INDEX'] = 'index.faiss'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
!uv run python -m anglican_search.embed_library --phase all --encode-batch 512

## 6. Verify the finished index


In [ ]:
!uv run python -c "import faiss; print('vectors:', faiss.read_index('index.faiss').ntotal)"

## 7. (Optional) quick sanity search

Skips the reranker (`--no-rerank`) to stay fast; downloads only the embedder.


In [ ]:
!uv run python -m anglican_search.search --q "the doctrine of the Holy Trinity" --k 3 --no-rerank

## 8. Retrieve the index

Download `index.faiss` (and keep `library.db`) from the file browser, then deploy both
to the CPU serving box.
